# Competition Simulation 2: Feature Iteration with Ablations

This notebook demonstrates a competition-realistic feature engineering loop:
- formulate hypotheses,
- run CV experiments,
- keep an explicit decision log,
- and only promote changes that materially improve the score.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 1) Generate a reproducible dataset

We use synthetic data with known nonlinear structure and temporal effects so feature iterations are meaningful without external downloads.


In [3]:
n_total = 3400
n_train = 2700
t = np.arange(n_total)

x1 = rng.normal(size=n_total)
x2 = rng.normal(size=n_total)
x3 = rng.normal(size=n_total)
x4 = rng.normal(size=n_total)
x5 = rng.normal(size=n_total)   # weak signal
x6 = rng.normal(size=n_total)   # mostly noise
noise_a = rng.normal(size=n_total)
noise_b = rng.normal(size=n_total)

base_df = pd.DataFrame(
    {
        "time_idx": t,
        "x1": x1,
        "x2": x2,
        "x3": x3,
        "x4": x4,
        "x5": x5,
        "x6": x6,
        "noise_a": noise_a,
        "noise_b": noise_b,
    }
)

base_df["sin_time"] = np.sin(t / 24)
base_df["cos_time"] = np.cos(t / 24)

# Candidate engineered features (all available at inference time)
base_df["x1_x2"] = base_df["x1"] * base_df["x2"]
base_df["x3_sq"] = base_df["x3"] ** 2
base_df["x4_abs"] = base_df["x4"].abs()
base_df["x1_roll7"] = base_df["x1"].rolling(7, min_periods=1).mean().shift(1)
base_df["x2_roll14"] = base_df["x2"].rolling(14, min_periods=1).mean().shift(1)
base_df[["x1_roll7", "x2_roll14"]] = base_df[["x1_roll7", "x2_roll14"]].fillna(0)

target = (
    2.1 * x1
    - 1.4 * x2
    + 1.2 * (x1 * x2)
    + 0.9 * (x3**2)
    + 0.45 * base_df["sin_time"].to_numpy()
    + 0.15 * x5
    + 0.0028 * t
    + rng.normal(0, 1.1, n_total)
)

base_df["target"] = target

train_df = base_df.iloc[:n_train].copy()
holdout_df = base_df.iloc[n_train:].copy()

X_train_full = train_df.drop(columns=["target"])
y_train = train_df["target"].to_numpy()

X_holdout_full = holdout_df.drop(columns=["target"])
y_holdout = holdout_df["target"].to_numpy()

print(f"Train rows: {len(train_df)}, Holdout rows: {len(holdout_df)}")
train_df.head()

Train rows: 2700, Holdout rows: 700


,time_idx,x1,x2,x3,x4,x5,x6,noise_a,noise_b,sin_time,cos_time,x1_x2,x3_sq,x4_abs,x1_roll7,x2_roll14,target
0,0,0.304717,0.398982,-0.477207,-1.144404,-0.762266,0.470372,-0.038106,-1.251170,0.000000,1.000000,0.121577,0.227726,1.144404,0.000000,0.000000,1.685991
1,1,-1.039984,0.824938,1.050534,1.036795,1.100795,0.740274,1.567228,1.385604,0.041655,0.999132,-0.857922,1.103621,1.036795,0.304717,0.398982,-3.738595
2,2,0.750451,-0.713047,-0.123180,-0.554001,0.549681,0.110448,-0.756832,-0.337251,0.083237,0.996530,-0.535107,0.015173,0.554001,-0.367634,0.611960,1.976337
3,3,0.940565,1.051855,0.611579,-2.413612,0.974537,1.003381,0.799052,-0.972093,0.124675,0.992198,0.989338,0.374029,2.413612,0.005061,0.170291,2.151676
4,4,-1.951035,-1.515692,0.671012,1.765356,-0.091879,-0.201259,0.316442,0.544661,0.165896,0.986143,2.957168,0.450258,1.765356,0.238937,0.390682,2.006884


## 2) Evaluation harness

We keep the model fixed and only change feature sets, so improvements are attributable to feature work.


In [4]:
cv = TimeSeriesSplit(n_splits=5)

def evaluate_feature_set(feature_cols):
    fold_rmse = []
    for tr_idx, va_idx in cv.split(X_train_full):
        model = GradientBoostingRegressor(
            n_estimators=280,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.9,
            random_state=RANDOM_SEED,
        )
        model.fit(X_train_full.iloc[tr_idx][feature_cols], y_train[tr_idx])
        pred = model.predict(X_train_full.iloc[va_idx][feature_cols])
        fold_rmse.append(np.sqrt(mean_squared_error(y_train[va_idx], pred)))
    return float(np.mean(fold_rmse)), float(np.std(fold_rmse, ddof=0))

## 3) Iterative ablations + decision log

Ablation mindset:
- Add promising features,
- remove likely-noisy ones,
- keep only changes that clear a minimum improvement threshold.


In [ ]:
base_features = [
    "time_idx", "x1", "x2", "x3", "x4", "x5", "x6", "noise_a", "noise_b", "sin_time", "cos_time"
]
nonlinear_features = base_features + ["x1_x2", "x3_sq", "x4_abs"]
with_roll_features = nonlinear_features + ["x1_roll7", "x2_roll14"]
ablate_noise_features = [c for c in with_roll_features if c not in ["x6", "noise_a", "noise_b"]]
ablate_calendar_features = [c for c in ablate_noise_features if c not in ["sin_time", "cos_time"]]

experiment_plan = [
    {
        "run_id": "01_baseline_raw",
        "features": base_features,
        "hypothesis": "Raw numeric + calendar features provide a stable baseline.",
    },
    {
        "run_id": "02_add_nonlinear_terms",
        "features": nonlinear_features,
        "hypothesis": "x1*x2 and squared terms should capture known nonlinear signal.",
    },
    {
        "run_id": "03_add_causal_rollups",
        "features": with_roll_features,
        "hypothesis": "Lagged rolling means may help with temporal drift.",
    },
    {
        "run_id": "04_ablate_noise_columns",
        "features": ablate_noise_features,
        "hypothesis": "Dropping noisy columns should reduce variance.",
    },
    {
        "run_id": "05_ablate_calendar_terms",
        "features": ablate_calendar_features,
        "hypothesis": "If seasonality adds little, removing calendar terms can simplify the model.",
    },
]

decision_rows = []
best_rmse = np.inf
best_run = None
best_features = None

for exp in experiment_plan:
    rmse_mean, rmse_std = evaluate_feature_set(exp["features"])
    improvement = (best_rmse - rmse_mean) if np.isfinite(best_rmse) else np.nan

    if not np.isfinite(best_rmse):
        decision = "promote"
        reason = "First run establishes the baseline."
        best_rmse = rmse_mean
        best_run = exp["run_id"]
        best_features = exp["features"]
    elif improvement > 0.01:
        decision = "promote"
        reason = f"Improves CV RMSE by {improvement:.4f}."
        best_rmse = rmse_mean
        best_run = exp["run_id"]
        best_features = exp["features"]
    else:
        decision = "reject"
        reason = f"No material gain vs best ({best_rmse:.4f})."

    decision_rows.append(
        {
            "run_id": exp["run_id"],
            "n_features": len(exp["features"]),
            "rmse_mean": rmse_mean,
            "rmse_std": rmse_std,
            "delta_vs_best_before_run": improvement,
            "decision": decision,
            "reason": reason,
            "hypothesis": exp["hypothesis"],
        }
    )

decision_log = pd.DataFrame(decision_rows)
decision_log

## 4) Inspect decisions and promote the champion

The decision log is the artifact you would keep in a real competition notebook/experiment tracker.


In [ ]:
decision_view = decision_log[["run_id", "n_features", "rmse_mean", "rmse_std", "decision", "reason"]].copy()
decision_view.sort_values("rmse_mean")

In [ ]:
champion_model = GradientBoostingRegressor(
    n_estimators=280,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    random_state=RANDOM_SEED,
)
champion_model.fit(X_train_full[best_features], y_train)
holdout_pred = champion_model.predict(X_holdout_full[best_features])
holdout_rmse = np.sqrt(mean_squared_error(y_holdout, holdout_pred))

print(f"Champion run: {best_run}")
print(f"Selected feature count: {len(best_features)}")
print(f"Holdout RMSE (single split sanity check): {holdout_rmse:.4f}")

feature_importance = pd.Series(champion_model.feature_importances_, index=best_features).sort_values(ascending=False)
feature_importance.head(12)

In [ ]:
feature_importance.head(12).sort_values().plot(kind="barh", figsize=(7, 4), title="Champion feature importances")
plt.tight_layout()

### Feature-iteration takeaway

Feature work is most useful when the loop is explicit: hypothesis → CV result → keep/reject decision.
Ablations prevent accidental complexity and make progress auditable.
